# Model Comparison — Thai Digits 51–55

Trains and compares 6 sklearn classifiers on the same dataset and augmentation pipeline used by the RF notebook.

| Model | Notes |
|-------|-------|
| Decision Tree | interpretable baseline |
| K-Nearest Neighbors | distance-based |
| SVM (RBF) | strong on small datasets |
| Naive Bayes | probabilistic baseline |
| Random Forest | ensemble of trees |
| MLP | shallow neural net (sklearn) |

Each model is evaluated on the same held-out validation set and 5-fold CV.  
The best model is saved as `best_model.joblib` for upload to the Flask app.

In [ ]:
!pip -q install scikit-learn joblib pillow matplotlib seaborn

## 1. Upload / locate dataset

In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()          # upload dataset.zip
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall('.')

DATA_DIR = 'dataset'
print('classes found:', sorted(os.listdir(DATA_DIR)))

## 2. Config & imports

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter
import joblib

from torchvision import transforms

from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix,
)

IMG_SIZE      = 28
VAL_SPLIT     = 0.15
AUG_PER_IMAGE = 10
SEED          = 42
CLASS_NAMES   = sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)))
assert CLASS_NAMES == ['51','52','53','54','55'], CLASS_NAMES
print('classes:', CLASS_NAMES)

## 3. Load images → flat pixel vectors

In [ ]:
label_to_idx = {c: i for i, c in enumerate(CLASS_NAMES)}

samples = []
for c in CLASS_NAMES:
    for path in glob.glob(os.path.join(DATA_DIR, c, '*')):
        try:
            img = Image.open(path).convert('L').resize((IMG_SIZE, IMG_SIZE))
            samples.append((img, label_to_idx[c]))
        except Exception as e:
            print('skipped', path, e)

print(f'total: {len(samples)}')
print('per-class:', {CLASS_NAMES[i]: c for i, c in Counter(y for _, y in samples).items()})

## 4. Offline augmentation (same pipeline as RF notebook)

In [ ]:
aug = transforms.Compose([
    transforms.RandomAffine(degrees=15, translate=(0.12, 0.12), scale=(0.75, 1.25), shear=8, fill=255),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.4, fill=255),
    transforms.RandomApply([transforms.ElasticTransform(alpha=20.0, sigma=4.0)], p=0.25),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.5, 1.2))], p=0.2),
])

def to_vec(pil_img):
    return np.asarray(pil_img, dtype=np.uint8).reshape(-1)

X_list, y_list = [], []
for img, label in samples:
    X_list.append(to_vec(img))
    y_list.append(label)
    for _ in range(AUG_PER_IMAGE):
        X_list.append(to_vec(aug(img)))
        y_list.append(label)

X = np.stack(X_list).astype(np.uint8)
y = np.asarray(y_list, dtype=np.int64)
print('X:', X.shape, '  y:', y.shape)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=VAL_SPLIT, stratify=y, random_state=SEED
)
print(f'train: {len(X_train)}   val: {len(X_val)}')

## 5. Define models

In [ ]:
MODELS = {
    'Decision Tree': DecisionTreeClassifier(
        max_depth=None, min_samples_leaf=1, random_state=SEED
    ),
    'KNN': KNeighborsClassifier(
        n_neighbors=5, metric='euclidean', n_jobs=-1
    ),
    'SVM (RBF)': SVC(
        kernel='rbf', C=10, gamma='scale', probability=True, random_state=SEED
    ),
    'Naive Bayes': GaussianNB(),
    'Random Forest': RandomForestClassifier(
        n_estimators=400, max_features='sqrt',
        class_weight='balanced', random_state=SEED, n_jobs=-1
    ),
    'MLP': MLPClassifier(
        hidden_layer_sizes=(256, 128), activation='relu',
        max_iter=300, random_state=SEED, early_stopping=True,
        validation_fraction=0.1, n_iter_no_change=15
    ),
}

## 6. Train & evaluate all models

In [ ]:
results = {}
trained = {}

for name, clf in MODELS.items():
    print(f'\n── {name} ──')
    t0 = time.time()
    clf.fit(X_train, y_train)
    train_sec = time.time() - t0

    y_pred = clf.predict(X_val)
    acc  = accuracy_score(y_val, y_pred)
    prec = precision_score(y_val, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_val, y_pred,    average='macro', zero_division=0)
    f1   = f1_score(y_val, y_pred,        average='macro', zero_division=0)

    results[name] = {'accuracy': acc, 'precision': prec, 'recall': rec,
                     'f1': f1, 'train_sec': train_sec}
    trained[name] = clf

    print(f'  Accuracy : {acc:.4f}')
    print(f'  Precision: {prec:.4f}  Recall: {rec:.4f}  F1: {f1:.4f}')
    print(f'  Train time: {train_sec:.1f}s')

## 7. Comparison table

In [ ]:
import pandas as pd

df = pd.DataFrame(results).T[['accuracy','precision','recall','f1','train_sec']]
df.columns = ['Accuracy','Precision','Recall','F1-score','Train (s)']
df = df.sort_values('Accuracy', ascending=False)

df_display = df.copy()
for col in ['Accuracy','Precision','Recall','F1-score']:
    df_display[col] = df_display[col].apply(lambda x: f'{x:.4f}')
df_display['Train (s)'] = df_display['Train (s)'].apply(lambda x: f'{float(x):.1f}')

print('\n=== Model Comparison ===')
print(df_display.to_string())

## 8. Bar chart comparison

In [ ]:
names   = list(results.keys())
metrics = ['accuracy', 'precision', 'recall', 'f1']
labels  = ['Accuracy', 'Precision', 'Recall', 'F1-score']
colors  = ['#2563eb', '#16a34a', '#d97706', '#dc2626']

x = np.arange(len(names))
width = 0.2

fig, ax = plt.subplots(figsize=(13, 5))
for i, (metric, label, color) in enumerate(zip(metrics, labels, colors)):
    vals = [results[n][metric] for n in names]
    bars = ax.bar(x + i * width, vals, width, label=label, color=color, alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{v:.2f}', ha='center', va='bottom', fontsize=7.5)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(names, rotation=15, ha='right')
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score')
ax.set_title('Model Comparison — Thai Digits 51–55')
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150)
plt.show()

## 9. Confusion matrices (all models)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.flatten()

for ax, (name, clf) in zip(axes, trained.items()):
    y_pred = clf.predict(X_val)
    cm  = confusion_matrix(y_val, y_pred)
    acc = accuracy_score(y_val, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                ax=ax, cbar=False)
    ax.set_title(f'{name}\nAcc={acc:.3f}', fontsize=11)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.suptitle('Confusion Matrices — Thai Digits 51–55', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. 5-Fold Cross-Validation (all models)

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_results = {}

for name, clf in MODELS.items():
    print(f'CV: {name}...', end=' ', flush=True)
    scores = cross_val_score(clf, X, y, cv=skf, scoring='accuracy', n_jobs=-1)
    cv_results[name] = scores
    print(f'{scores.mean():.4f} ± {scores.std():.4f}')

print('\n=== 5-Fold CV Summary (sorted by mean) ===')
print(f'{"Model":<20} {"Mean Acc":>10} {"± Std":>8}')
print('-' * 42)
for name, scores in sorted(cv_results.items(), key=lambda x: -x[1].mean()):
    print(f'{name:<20} {scores.mean():>10.4f} {scores.std():>8.4f}')

## 11. Save best model as `.joblib`

In [ ]:
best_name = max(results, key=lambda n: results[n]['f1'])
best_clf  = trained[best_name]
print(f'Best model: {best_name}  (F1={results[best_name]["f1"]:.4f})')

slug = best_name.lower().replace(' ','_').replace('(','').replace(')','').replace('/','_')
filename = f'best_model_{slug}.joblib'
bundle = {
    'model':       best_clf,
    'classes':     CLASS_NAMES,
    'image_shape': (IMG_SIZE, IMG_SIZE),
    'input':       'flatten 28x28 grayscale uint8 to 784-d vector, no normalization',
    'model_name':  best_name,
}
joblib.dump(bundle, filename, compress=3)
print(f'Saved → {filename}  ({os.path.getsize(filename)/1024:.1f} KB)')

from google.colab import files as gfiles
gfiles.download(filename)

## 12. (Optional) Save ALL models as one zip

In [ ]:
import zipfile as zf

os.makedirs('all_models', exist_ok=True)
saved = []
for name, clf in trained.items():
    slug = name.lower().replace(' ','_').replace('(','').replace(')','').replace('/','_')
    fname = f'all_models/{slug}.joblib'
    joblib.dump({'model': clf, 'classes': CLASS_NAMES, 'model_name': name,
                 'image_shape': (IMG_SIZE, IMG_SIZE),
                 'input': 'flatten 28x28 grayscale uint8 to 784-d vector, no normalization'},
                fname, compress=3)
    saved.append(fname)
    print(f'  {fname}  ({os.path.getsize(fname)/1024:.1f} KB)')

with zf.ZipFile('all_models.zip', 'w') as z:
    for f in saved:
        z.write(f)

gfiles.download('all_models.zip')
print('Downloaded all_models.zip')